# 🎯 Objetivos

El objetivo de este script es combinar ambos sistemas, de contenido y colaborativo, para construir un sistema híbrido que combine las ventajas de ambos.

* ⚙️ **Configuración inicial**. Rutas, paquetes y funciones.
* ♻️ **Carga de datos**.
* ✅ **Chequear la integridad**.
* 📈 **Evaluación del modelo**. Probar distintos pesos y elegir el mejor alpha.

# ⚙️ Configuración

## Rutas (paths)

In [1]:
import os

# Rutas en función de la carpeta raíz del proyecto (README.md)
base_path = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) if "__file__" in locals() else os.path.abspath("..")

# Subrutas
data_dir = os.path.join(base_path, "data")
db_path = os.path.join(data_dir, "steam.db")
src_dir = os.path.join(base_path, "src")
models_dir = os.path.join(base_path, "models")
models_content_dir = os.path.join(models_dir, "content")
models_collab_dir = os.path.join(models_dir, "collaborative")
results_dir = os.path.join(base_path, "res")

## Paquetes y funciones

In [ ]:
# Sistema y configuración
import warnings
warnings.filterwarnings("ignore")
import sqlite3
import glob

# Modelos
import joblib

# Análisis y manipulación de datos
import pandas as pd
import numpy as np

# Matrices y transformaciones
from scipy import sparse

# Funciones personalizadas    
from utils.evaluation_functions import evaluate_hybrid_LMO_at_k

# ♻️ Carga de datos

In [3]:
# Información del sistema de contenido

# Base escalada (con todas las variables)
conn = sqlite3.connect(db_path)
cur = conn.cursor()
games = pd.read_sql("SELECT * FROM games_escaled_final", sqlite3.connect(db_path))
conn.close()

games = games.copy()   # Hacemos una copia para trabajar

# Perfil de contenido normalizado por usuario (no se usa matriz precomputada)
U_norm = None
user_to_pos = None

# Representación vectorial final de los juegos (estructural + TF-IDF)   
X_combined = sparse.load_npz(
    os.path.join(models_content_dir, "X_combined_alpha_0.7.npz")   
)
    
# Índice de juegos   
idx = pd.read_csv(os.path.join(models_dir, "games_index.csv")) 

In [ ]:
# Información del sistema colaborativo

# Matriz usuario-juego original (binaria, con IDF ya aplicado según script 8 si corresponde)
R = sparse.load_npz(os.path.join(models_collab_dir, "R_user_based.npz"))

# Matriz usuario-juego normalizada  
R_norm = sparse.load_npz(os.path.join(models_collab_dir, "R_norm_user_based.npz"))

# Modelo kNN entrenado
knn_artifacts = joblib.load(os.path.join(models_collab_dir, "knn_user_based_70.0.pkl"))

knn = knn_artifacts["knn"]

# Diccionario de mapeo: user_id - fila en R_norm
user_map = knn_artifacts["user_map"]

# Diccionario de mapeo: índice de columna -> appid (mapa inverso)
game_map = knn_artifacts["game_map"]

# Diccionario de mapeo: indice invverso columna - appid
inv_game_map = knn_artifacts["inv_game_map"]


In [5]:
# Train y Test
conn = sqlite3.connect(db_path)
cur = conn.cursor()

user_train = pd.read_sql("SELECT * FROM user_train_LMO", sqlite3.connect(db_path))
user_test = pd.read_sql("SELECT * FROM user_test_LMO", sqlite3.connect(db_path))

conn.close()

# Hacemos una copia para trabajar
user_train = user_train.copy()
user_test = user_test.copy()

In [6]:
# Normalización de tipos
user_train["user_id"] = user_train["user_id"].astype(np.int64)
user_test["user_id"]  = user_test["user_id"].astype(np.int64)
user_train["appid"]   = user_train["appid"].astype(np.int64)
user_test["appid"]    = user_test["appid"].astype(np.int64)

# ✅ Chequeos

El objetivo es asegurar la coherencia de los datos importados para crear el sistema de recomendación híbrido.

In [7]:
# Dimensiones
print("games        :", games.shape)
print("idx          :", idx.shape)
print("X_combined   :", X_combined.shape)
print("R_norm       :", R_norm.shape)

# Contenido desajuste
assert X_combined.shape[0] == len(idx) == len(games), \
       "Desajuste filas entre X_combined, idx y games"

# Colaborativo deajuste
assert R_norm.shape[0] == len(user_map), "R_norm y user_map no tienen el mismo nº de usuarios"
assert R_norm.shape[1] == len(inv_game_map) == len(game_map), "R_norm y mapas de juegos no alinean"


games        : (2468, 15)
idx          : (2468, 2)
X_combined   : (2468, 20012)
R_norm       : (47627, 2468)


In [8]:
# Chequeos contenido - colaborativo - test/train

# Número de usuarios
users_train = set(user_train["user_id"])
users_test  = set(user_test["user_id"])
users_content = set(user_train["user_id"])
users_collab  = set(user_map.keys())

print("Usuarios train       :", len(users_train))
print("Usuarios test        :", len(users_test))
print("Usuarios contenido   :", len(users_content))
print("Usuarios colaborativo:", len(users_collab))


print("Usuarios comunes contenido ∩ colaborativo:",len(users_content & users_collab))
print("Usuarios train están en contenido:", users_train.issubset(users_content))
print("Usuarios train están en colaborativo:", users_train.issubset(users_collab))
print("Usuarios test están en contenido:", users_test.issubset(users_content))
print("Usuarios test están en colaborativo:", users_test.issubset(users_collab))

assert len(users_content & users_collab) > 0, "No hay intersección entre usuarios de ambos sistemas"
assert users_train.issubset(users_content), "Hay usuarios de train fuera de user_to_pos"
assert users_train.issubset(users_collab),  "Hay usuarios de train fuera de user_map"
assert users_test.issubset(users_content),  "Hay usuarios de test fuera de user_to_pos"
assert users_test.issubset(users_collab),   "Hay usuarios de test fuera de user_map"

Usuarios train       : 47627
Usuarios test        : 47627
Usuarios contenido   : 47627
Usuarios colaborativo: 47627
Usuarios comunes contenido ∩ colaborativo: 47627
Usuarios train están en contenido: True
Usuarios train están en colaborativo: True
Usuarios test están en contenido: True
Usuarios test están en colaborativo: True


In [9]:
# Número de juegos
games_catalog = set(games["appid"])
games_idx     = set(idx["appid"])
appid_to_row = {appid: i for i, appid in enumerate(idx["appid"].values)} 
games_content = set(appid_to_row.keys())
games_collab  = set(game_map.keys())  # columnas de R_norm

print("Juegos catálogo (games) :", len(games_catalog))
print("Juegos índice (idx)     :", len(games_idx))
print("Juegos contenido (X)    :", len(games_content))
print("Juegos colaborativo (R) :", len(games_collab))

# Consistencia interna contenido
assert games_catalog == games_idx, "games y idx tienen distinto conjunto de appid"
assert games_content == games_idx, "appid_to_row no coincide con idx/games"

# Colaborativo debe ser subconjunto del catálogo de contenido
missing_in_content = games_collab - games_catalog
print("Juegos colaborativo fuera del catálogo de contenido:", len(missing_in_content))
assert len(missing_in_content) == 0, "Hay appid en colaborativo que no están en games/idx"

# Comprobar que los índices son válidos
max_row_X = X_combined.shape[0] - 1
assert all(0 <= r <= max_row_X for r in appid_to_row.values()), "Algún índice de appid_to_row está fuera de rango"

max_col_R = R_norm.shape[1] - 1
assert inv_game_map.shape[0] == max_col_R + 1, "inv_game_map no cubre todas las columnas de R_norm"


Juegos catálogo (games) : 2468
Juegos índice (idx)     : 2468
Juegos contenido (X)    : 2468
Juegos colaborativo (R) : 2468
Juegos colaborativo fuera del catálogo de contenido: 0


In [10]:
# Nulos y tipos en train-test

assert user_train["user_id"].notna().all()
assert user_train["appid"].notna().all()
assert user_test["user_id"].notna().all()
assert user_test["appid"].notna().all()

assert np.issubdtype(user_train["user_id"].dtype, np.integer)
assert np.issubdtype(user_train["appid"].dtype, np.integer)
assert np.issubdtype(user_test["user_id"].dtype, np.integer)
assert np.issubdtype(user_test["appid"].dtype, np.integer)


# 📈 Evaluación del modelo

In [11]:
alphas = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

for alpha in alphas:
    evaluate_hybrid_LMO_at_k(
        k=10,
        alpha=alpha,
        user_train=user_train,
        user_test=user_test,
        U_norm=U_norm,
        user_to_pos=user_to_pos,
        X_combined=X_combined,
        R_norm=R_norm,
        R=R,
        user_map=user_map,
        game_map=game_map,
        inv_game_map=inv_game_map,
        games=games,
        idx=idx,
        knn=knn,
        results_dir=results_dir,
        tag="20251130",
        top_neighbors=50,
    )

In [12]:
alphas = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

for alpha in alphas:
    evaluate_hybrid_LMO_at_k(
        k=50,
        alpha=alpha,
        user_train=user_train,
        user_test=user_test,
        U_norm=U_norm,
        user_to_pos=user_to_pos,
        X_combined=X_combined,
        R_norm=R_norm,
        R=R,
        user_map=user_map,
        game_map=game_map,
        inv_game_map=inv_game_map,
        games=games,
        idx=idx,
        knn=knn,
        results_dir=results_dir,
        tag="20251130",
        top_neighbors=50,
    )

# Mejor alfa

In [ ]:
def summary_best_alpha(results_dir, k, primary="recall@k", secondary="ndcg@k"):
    pattern = os.path.join(results_dir, f"metrics_hybrid_k{k}_a*.csv")
    files = sorted(glob.glob(pattern))

    dfs = [pd.read_csv(f) for f in files]
    metrics = pd.concat(dfs, ignore_index=True)

    # Orden: primero métrica principal, luego secundaria, y como desempate item_coverage@k
    metrics_sorted = metrics.sort_values(
        [primary, secondary, "item_coverage@k"],
        ascending=[False, False, False],
    ).reset_index(drop=True)

    best = metrics_sorted.iloc[0]

    print(metrics_sorted[["alpha", "precision@k", "recall@k", "f1@k", "ndcg@k", "hit_rate@k", "item_coverage@k"]])
    print("\nMejor alfa según", primary, f"(desempate con {secondary} y cobertura):")
    print(best)

    return metrics_sorted, best

In [14]:
metrics_k10, best_k10 = summary_best_alpha(results_dir, k=10)

    alpha  precision@k  recall@k      f1@k    ndcg@k  hit_rate@k  \
0     0.2     0.073782  0.245939  0.113510  0.355743    0.546854   
1     0.1     0.073517  0.245057  0.113103  0.355526    0.545951   
2     0.3     0.073318  0.244392  0.112796  0.352629    0.543620   
3     0.0     0.072849  0.242831  0.112076  0.352194    0.542465   
4     0.4     0.072199  0.240662  0.111075  0.345319    0.537363   
5     0.5     0.068757  0.229191  0.105780  0.329238    0.516010   
6     0.6     0.061350  0.204499  0.094384  0.296644    0.471917   
7     0.7     0.048223  0.160742  0.074189  0.232103    0.384530   
8     0.8     0.031239  0.104129  0.048059  0.151207    0.258257   
9     0.9     0.018206  0.060687  0.028009  0.089840    0.151658   
10    1.0     0.011273  0.037577  0.017343  0.057018    0.094547   

    item_coverage@k  
0          0.628849  
1          0.617504  
2          0.649919  
3          0.607374  
4          0.674230  
5          0.696110  
6          0.698541  
7      

In [15]:
metrics_k50, best_k50 = summary_best_alpha(results_dir, k=50)

    alpha  precision@k  recall@k      f1@k    ndcg@k  hit_rate@k  \
0     0.2     0.028663  0.477719  0.054081  0.397572    0.829949   
1     0.1     0.028543  0.475718  0.053855  0.397167    0.828606   
2     0.3     0.028527  0.475445  0.053824  0.395044    0.827682   
3     0.0     0.028330  0.472162  0.053452  0.394518    0.825645   
4     0.4     0.027882  0.464701  0.052608  0.388497    0.818695   
5     0.5     0.026481  0.441346  0.049964  0.375103    0.798539   
6     0.6     0.023661  0.394356  0.044644  0.345186    0.751339   
7     0.7     0.019166  0.319427  0.036162  0.284925    0.659878   
8     0.8     0.013708  0.228470  0.025865  0.203153    0.515317   
9     0.9     0.008517  0.141957  0.016071  0.128591    0.340899   
10    1.0     0.005353  0.089214  0.010100  0.082433    0.219980   

    item_coverage@k  
0          0.872771  
1          0.852917  
2          0.895867  
3          0.840357  
4          0.917747  
5          0.934765  
6          0.943679  
7      